Helper function

In [ ]:
# Data: ../csv/FigS2_D_E_F_low_avalanche_example_avalanche_sizeduradcc_/  (the original loading code is kept below each data line, as comments)
import os, glob, pickle, re, sys   # standard imports
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'analysis_scripts'))   # the criticality library
import pandas as pd
import numpy as np
CSV = Path('..') / 'csv' / 'FigS2_D_E_F_low_avalanche_example_avalanche_sizeduradcc_'
OUT = Path('..') / 'figure_output'
OUT.mkdir(exist_ok=True)


def read_csv(name, **kw):
    """A table of this notebook (round_trip: the floats exactly as written)."""
    return pd.read_csv(CSV / name, float_precision='round_trip', **kw)

In [ ]:
import numpy as np

def get_firing_matrix(firing_rates_dict):
    """
    Extracts and transposes the firing rate matrix from a dictionary.
    Returns a matrix of shape (T_bins, N_units).
    """
    all_units = list(firing_rates_dict.keys())
    firing_matrix = np.array([firing_rates_dict[unit]['firing_rate'] for unit in all_units])
    return firing_matrix.T  # (T_bins, N_units)
def compute_firing_rates(st_time, starttime, endtime, bin_size=1.0):
    """
    Computes firing rates for multiple units using vectorized operations.

    Parameters:
        st_time (list of np.ndarray): A list where each entry is an array of spike times for a unit.
        starttime (float): The start time of the epoch.
        endtime (float): The end time of the epoch.
        bin_size (float, optional): The size of the bins in seconds. Default is 1.0.

    Returns:
        dict: A dictionary where each key is 'unitX' and values contain firing rates and time bin centers.
    """
    # Generate time bins
    bins = np.arange(starttime, endtime + bin_size, bin_size)
    time_bin_centers = bins[:-1] + (bin_size / 2)  # Compute the center of each time bin

    # Convert spike times list to a NumPy array (object dtype for jagged arrays)
    spike_times_array = np.array(st_time, dtype=object)

    # Compute spike counts using vectorized histogram
    spike_counts = np.array([
        np.histogram(spike_times[(spike_times >= starttime) & (spike_times < endtime)], bins=bins)[0]
        for spike_times in spike_times_array
    ])

    # Calculate firing rates (spike count per second)
    firing_rates = spike_counts / bin_size  # Shape: (num_units, num_bins)

    # Store results in a dictionary
    firing_rates_dict = {
        f'unit{idx + 1}': {'firing_rate': firing_rates[idx], 'time_stamps': time_bin_centers}
        for idx in range(len(spike_times_array))
    }

    return firing_rates_dict
import numpy as np

def filter_nrem_hse_fast(firing_rates_dict, NREM_HSE_epochs):
    """
    Filters firing rates to keep only bins within NREM_HSE epochs (optimized for speed).

    Parameters:
        firing_rates_dict (dict): Output from `compute_firing_rates`, containing firing rates and time stamps.
        NREM_HSE_epochs (np.ndarray): An array of shape (num_epochs, 2) where each row is [start, stop] time.

    Returns:
        dict: Filtered version of `firing_rates_dict`, keeping only time bins within NREM_HSE epochs.
    """
    filtered_firing_rates = {}

    # Sort NREM_HSE_epochs for faster searching
    sorted_epochs = np.sort(NREM_HSE_epochs, axis=0)

    for unit, data in firing_rates_dict.items():
        firing_rate = data['firing_rate']
        time_stamps = data['time_stamps']

        # Fast lookup: Find which epochs each time bin belongs to
        idx_starts = np.searchsorted(sorted_epochs[:, 0], time_stamps, side="right") - 1
        idx_ends = np.searchsorted(sorted_epochs[:, 1], time_stamps, side="left")

        # A valid time bin is inside an epoch if the start index == end index
        mask = (idx_starts == idx_ends) & (idx_starts >= 0) & (idx_starts < len(sorted_epochs))

        # Apply mask to filter time bins
        filtered_firing_rates[unit] = {
            'firing_rate': firing_rate[mask],
            'time_stamps': time_stamps[mask]
        }

    return filtered_firing_rates
def compute_ripple_rate(ripple_timestamps, bin_centers, bin_size=1.0):
    """
    Computes ripple rate (ripples per second) in each bin.

    Parameters:
        ripple_timestamps (np.ndarray): 1D array of ripple event times (e.g., ripple peaks).
        bin_centers (np.ndarray): 1D array of time points corresponding to the center of each bin.
        bin_size (float): Bin size in seconds.

    Returns:
        np.ndarray: Ripple rate (Hz) per bin.
    """
    half_bin = bin_size / 2.0
    # Construct bin edges from centers
    bin_edges = np.concatenate(([bin_centers[0] - half_bin], bin_centers + half_bin))

    ripple_counts, _ = np.histogram(ripple_timestamps, bins=bin_edges)
    return ripple_counts / bin_size  # ripple rate in Hz
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd
def compute_population_synchrony_from_hist(E_spikes, bin_size=0.025):
    """
    Compute population synchrony from spike trains by:
    1. Binning into 25ms bins using np.histogram
    2. Aggregating every 2 bins (50ms) to compute synchrony

    Parameters:
        E_spikes (list of np.arrays): spike times for each unit
        bin_size (float): bin size in seconds (default 25ms)

    Returns:
        pd.DataFrame: columns ['time', 'synchrony']
    """
    num_units = len(E_spikes)

    # Get global time range
    all_spikes = np.concatenate(E_spikes)
    t_min = np.min(all_spikes)
    t_max = np.max(all_spikes)

    # Bin edges for 25ms
    bin_edges = np.arange(t_min, t_max + bin_size, bin_size)
    n_bins = len(bin_edges) - 1

    # Fill firing rate matrix
    firing_matrix = np.zeros((num_units, n_bins), dtype=int)
    for unit_idx, spikes in enumerate(E_spikes):
        counts, _ = np.histogram(spikes, bins=bin_edges)
        firing_matrix[unit_idx] = counts

    # Compute synchrony across every 50ms (2-bin) chunk
    step = 2
    sync = []
    time_centers = []

    for i in range(0, n_bins - 1, step):
        activity = np.sum(firing_matrix[:, i:i+2], axis=1)
        frac_active = np.sum(activity > 0) / num_units
        sync.append(frac_active)
        time_centers.append((bin_edges[i] + bin_edges[i+2]) / 2)

    return pd.DataFrame({
        'time': time_centers,
        'synchrony': sync
    })

Loading everything

In [ ]:


import matplotlib.pyplot as plt
import nelpy as nel
import nelpy.plotting as npl
import numpy as np
import seaborn as sns
from neuro_py.io import loading
from skimage import measure
import os

import logging

logging.getLogger().setLevel(logging.ERROR)
basepath = r"W:\data\PPP\PPP20\day4"
basename=os.path.basename(basepath)# Define the new folder path
#create working folder
new_folder_path = os.path.join(basepath, "complex_system")
brain_region="CA1"
animal="mouse"   #or Mouse
cell_type='all'
path_parts = basepath.split("\\")  # Split the path using backslash
animal_name = path_parts[-2]  # Second-to-last part is the animal name
day = path_parts[-1]  # Last part is the day
# ---- data (../csv) ----
from types import SimpleNamespace
save_dir = str(OUT)
nrem_epochs = nel.EpochArray(read_csv('states_NREM.csv')[['start', 'stop']].to_numpy())
awake_epochs = nel.EpochArray(read_csv('states_WAKE.csv')[['start', 'stop']].to_numpy())
rem_epochs = nel.EpochArray(read_csv('states_REM.csv')[['start', 'stop']].to_numpy())
epoch_df = read_csv('epochs.csv')
sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
# raw spikes are not shipped: the avalanches they give are read inside run_dcc_pipeline_forplot
st = SimpleNamespace(time=None)
spike_times = None
# ---- the same from the recordings, with analysis_scripts/: ----
# save_dir = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\avalanche"
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)
# # Create the folder if it doesn't exist
# if not os.path.exists(new_folder_path):
#     os.makedirs(new_folder_path)
#     print(f"Folder created: {new_folder_path}")
# else:
#     print(f"Folder already exists: {new_folder_path}")
# # load in spike data from hpc pyramidal cells
# st, cm = loading.load_spikes(
#     basepath, brainRegion=brain_region ,                    # putativeCellType="Pyr"
# )
# # load theta epochs
# state_dict = loading.load_SleepState_states(basepath)
# nrem_epochs = nel.EpochArray(
#     state_dict["NREMstate"],
# )
# awake_epochs = nel.EpochArray(
#     state_dict["WAKEstate"],
# )
# rem_epochs = nel.EpochArray(
#     state_dict["REMstate"],
# )
# import scipy.io
# """ file_path = os.path.join(basepath, f"HSE_epochs.mat")
# mat_data = scipy.io.loadmat(file_path)
# HSE_epochs = mat_data['HSE_epochs'] """
# epoch_df = loading.load_epoch(basepath)
# sleep_epochs = nel.EpochArray(np.array([epoch_df.startTime, epoch_df.stopTime]).T)
# spike_times = st.time # From your code
# E_spikes = st.time[(cm.putativeCellType == 'Pyramidal Cell')]
# E_CellID=cm.UID[(cm.putativeCellType == 'Pyramidal Cell')]
# E_firingrate=cm.firingRate[(cm.putativeCellType == 'Pyramidal Cell')]
# I_spikes= st.time[(cm.putativeCellType.str.contains("Interneuron"))]
# I_CellID=cm.UID[(cm.putativeCellType.str.contains("Interneuron"))]

In [ ]:
epoch_df

In [ ]:
import numpy as np
import scipy.io as sio  
# Load ripple events
# ---- data (../csv) ----
ripple_events = read_csv('ripple_events.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# ripple_events = loading.load_ripples_events(basepath=basepath)
ripple_epochs = nel.EpochArray(np.array([ripple_events.peaks, ripple_events.peaks]).T)
    # Load barrage events

# ---- data (../csv) ----
# barrage, delta-wave and all_d2 files: loaded by the original notebook, not used by its panels
barrage_epochs, delta_events, all_d2 = [], [], []
# ---- the same from the recordings, with analysis_scripts/: ----
# try:
#     # Load barrage events
#     #brpath=r"W:\data\PPP\CA2\PPP7\day8"
#     barrage_events = loading.load_barrage_events(basepath=basepath)  
#     # Create barrage epochs
#     barrage_epochs = nel.EpochArray(np.array([barrage_events.start, barrage_events.stop]).T)
#     # Find barrages in NREM epochs
#     barrages_in_nrem = barrage_epochs & nrem_epochs
#     mat_path = os.path.join(basepath, f'{basename}.HSEn2.events.mat')
# 
#     data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
#     barrage = data['HSEn2']
#     barrage_timestamps= barrage.timestamps 
#     # Extract start times
# except Exception as e:
#     # General exception for unexpected errors
#     print(f"No barrage file: {e}")
#     barrage_epochs=[]
# try:
#     # Load barrage events
#     # Create barrage epochs
#     mat_path = os.path.join(basepath, f'{basename}.deltaWaves.events.mat')
# 
#     data = scipy.io.loadmat(mat_path, struct_as_record=False, squeeze_me=True)
#     deltaWaves = data['deltaWaves']
#     delta_events= deltaWaves.timestamps 
#     # Extract start times
# except Exception as e:
#     # General exception for unexpected errors
#     print(f"No barrage file: {e}")
#     delta_events=[]
# try:# Step 2: Get the common time bins (already defined)
#     matfile = os.path.join(basepath, "all_d2.mat")
#     matdata = sio.loadmat(matfile)
#     # Step 2: Get the common time bins (already defined)
#     # Assuming the variable is named 'all_d2' inside the file:
#     all_d2 = matdata["all_d2"]
# except Exception as e:
#     # General exception for unexpected errors
#     print(f"No d2 file: {e}")
#     all_d2 = []
# Extract pre-sleep epoch
# Example: pre-sleep indices
pre_sleep_idx = [0]
# Collect all intervals from those indices
pre_intervals = np.array([
    [epoch_df.startTime[i], epoch_df.stopTime[i]]
    for i in pre_sleep_idx
])
# Build EpochArray with multiple rows
pre_epoch = nel.EpochArray(pre_intervals)
# Overlap with state epochs
pre_NREM  = pre_epoch & nrem_epochs[nrem_epochs.durations > 0]
pre_awake = pre_epoch & awake_epochs[awake_epochs.durations > 0]
pre_REM   = pre_epoch & rem_epochs[rem_epochs.durations > 0]
# Extract middle sleep epoch
mid_sleep_idx = [2]
mid_intervals = np.array([[epoch_df.startTime[i], epoch_df.stopTime[i]] for i in mid_sleep_idx])
mid_epoch = nel.EpochArray(mid_intervals)
mid_NREM = mid_epoch & nrem_epochs[nrem_epochs.durations > 0]  # Filter by NREM duration
mid_awake=mid_epoch & awake_epochs[awake_epochs.durations > 0] 
mid_REM=mid_epoch & rem_epochs[rem_epochs.durations > 0] 
# Extract post-sleep epoch
post_sleep_idx = [4]
post_intervals = np.array([[epoch_df.startTime[i], epoch_df.stopTime[i]] for i in post_sleep_idx])
post_epoch = nel.EpochArray(post_intervals)
post_NREM = post_epoch & nrem_epochs[nrem_epochs.durations >0]  # Compute overlapping epochs
post_awake=post_epoch & awake_epochs[awake_epochs.durations > 0] 
post_REM=post_epoch & rem_epochs[rem_epochs.durations > 0] 
def adjust_intervals(intervals, target):
    """
    Adjust intervals to sum to exactly the target duration.
    
    If the accumulated duration plus the next interval exceeds the target,
    a partially trimmed interval is added.
    """
    adjusted = []
    accumulated = 0
    for start, end in intervals:
        duration = end - start
        if accumulated + duration < target:
            adjusted.append((start, end))
            accumulated += duration
        else:
            # Calculate remaining time needed
            remaining = target - accumulated
            if remaining > 0:
                # Append the trimmed interval
                adjusted.append((start, start + remaining))
            break
    return adjusted

# Compute the total durations
total_pre = sum(pre_NREM.durations)
total_mid = sum(mid_NREM.durations)
total_post = sum(post_NREM.durations)

# Set the target duration to the smallest total duration
target_duration = max(total_pre, total_mid, total_post)

# Get NREM intervals
pre_intervals = pre_NREM.data
mid_intervals = mid_NREM.data
post_intervals = post_NREM.data

# Adjust all sets of intervals
adjusted_pre_intervals = adjust_intervals(pre_intervals, target_duration)
adjusted_mid_intervals = adjust_intervals(mid_intervals, target_duration)
adjusted_post_intervals = adjust_intervals(post_intervals, target_duration)

# Convert adjusted intervals into EpochArrays
pre_NREMs = nel.EpochArray(np.array(adjusted_pre_intervals))
mid_NREMs = nel.EpochArray(np.array(adjusted_mid_intervals))
post_NREMs = nel.EpochArray(np.array(adjusted_post_intervals))

# Find ripples occurring within each adjusted epoch
ripples_in_pre = ripple_epochs & pre_NREMs
ripples_in_mid = ripple_epochs & mid_NREMs
ripples_in_post = ripple_epochs & post_NREMs
if isinstance(barrage_epochs, nel.EpochArray):
    barrages_in_pre = barrage_epochs & pre_NREMs
    barrages_in_mid = barrage_epochs & mid_NREMs
    barrages_in_post = barrage_epochs & post_NREMs
else:
    barrages_in_pre = barrages_in_mid = barrages_in_post = None
# Print final durations for verification
# matfile = os.path.join(basepath, 'swslope.mat')
# data = scipy.io.loadmat(matfile)
# swslope_array = data['swslope']
##sia_data = scipy.io.loadmat(sia_event_path)
#SIA_periods = sia_data['SIA_periods']

Bin ripples

In [ ]:
import numpy as np
import pandas as pd
import umap
import plotly.express as px
from scipy.ndimage import gaussian_filter1d
from scipy.stats import zscore

# ----------------------------------------------------------
# Loop over different bin sizes (assuming bin_sizes is defined)
# ----------------------------------------------------------
bin_sizes = [1]
sigma = 1
starttime = 0
endtime = epoch_df.stopTime.iloc[-1]  # From your code

# ---- data (../csv) ----
_r = read_csv('swr_rate_1s_pre.csv'); pre_timestamps, pre_ripple_rate = _r['pre_timestamps'].to_numpy(), _r['pre_ripple_rate'].to_numpy()
_r = read_csv('swr_rate_1s_mid.csv'); mid_timestamps, mid_ripple_rate = _r['mid_timestamps'].to_numpy(), _r['mid_ripple_rate'].to_numpy()
_r = read_csv('swr_rate_1s_post.csv'); post_timestamps, post_ripple_rate = _r['post_timestamps'].to_numpy(), _r['post_ripple_rate'].to_numpy()
# ---- the same from the recordings, with analysis_scripts/: ----
# for current_bin_size in bin_sizes:
#     print(f"\n=== Processing bin_size = {current_bin_size} ===")
# 
#     # 1) Compute firing rates for the current bin size
#     firing_rates = compute_firing_rates(spike_times, starttime, endtime, current_bin_size)
# 
#     # 2) Smooth and z-score firing rates
#     smoothed_zscored_rates = {}
#     for unit, data in firing_rates.items():
#         rates = data['firing_rate']
#         smoothed_rates = gaussian_filter1d(rates, sigma=sigma)  # Smooth with Gaussian kernel
#         zscored_rates = zscore(smoothed_rates)  # Z-score normalization
# 
#         smoothed_zscored_rates[unit] = {
#             'firing_rate': zscored_rates,
#             'time_stamps': data['time_stamps']
#         }
# 
#     # 3) Filter into pre-, middle-, and post-sleep epochs
#     pre_filtered_firing_rates = filter_nrem_hse_fast(smoothed_zscored_rates, adjusted_pre_intervals)
#     mid_filtered_firing_rates = filter_nrem_hse_fast(smoothed_zscored_rates, adjusted_mid_intervals)
#     post_filtered_firing_rates = filter_nrem_hse_fast(smoothed_zscored_rates, adjusted_post_intervals)
# 
#     # 4) Extract firing rate matrices
#     pre_firing_matrix = get_firing_matrix(pre_filtered_firing_rates)
#     mid_firing_matrix = get_firing_matrix(mid_filtered_firing_rates)
#     post_firing_matrix = get_firing_matrix(post_filtered_firing_rates)
# 
#     # 5) Concatenate for classification/embedding
#     X = np.concatenate((pre_firing_matrix, mid_firing_matrix, post_firing_matrix), axis=0)
#     y = np.concatenate((
#         np.zeros(pre_firing_matrix.shape[0]),  # Label 0 for pre
#         np.ones(mid_firing_matrix.shape[0]),   # Label 1 for mid
#         np.full(post_firing_matrix.shape[0], 2)  # Label 2 for post
#     ))
# 
#     pre_timestamps = pre_filtered_firing_rates[list(pre_filtered_firing_rates.keys())[0]]['time_stamps']
#     mid_timestamps = mid_filtered_firing_rates[list(mid_filtered_firing_rates.keys())[0]]['time_stamps']
#     post_timestamps = post_filtered_firing_rates[list(post_filtered_firing_rates.keys())[0]]['time_stamps']
# 
#     # Compute ripple rates
#     pre_ripple_rate = compute_ripple_rate(ripples_in_pre.data, pre_timestamps)
#     mid_ripple_rate = compute_ripple_rate(ripples_in_mid.data, mid_timestamps)
#     post_ripple_rate = compute_ripple_rate(ripples_in_post.data, post_timestamps)


high ripple and low ripple avalanche

In [ ]:
import numpy as np

def get_ripple_intervals(bin_centers, ripple_rate, bin_size, threshold=0.1, min_consecutive_bins=1):
    """
    Finds consecutive intervals where the ripple rate is below a given threshold (low ripple intervals)
    and consecutive intervals where the ripple rate is at or above the threshold (high ripple intervals).

    Parameters:
        bin_centers (np.ndarray): 1D array of time points corresponding to the centers of bins.
        ripple_rate (np.ndarray): 1D array of ripple rates (Hz) corresponding to each bin.
        bin_size (float): Size of each time bin in seconds.
        threshold (float): Ripple rate threshold (Hz). Bins with ripple_rate < threshold are considered low.
        min_consecutive_bins (int): Minimum number of consecutive bins to form a valid interval.
        
    Returns:
        low_intervals (list of tuples): Each tuple is (start_time, end_time) of a low ripple interval.
        high_intervals (list of tuples): Each tuple is (start_time, end_time) of a high ripple interval.
    """
    low_intervals = []
    high_intervals = []
    
    current_low_start = None
    current_high_start = None

    for i, rate in enumerate(ripple_rate):
        if rate < threshold:
            # If we are in a high interval, close it.
            if current_high_start is not None:
                if (i - current_high_start) >= min_consecutive_bins:
                    start_time = bin_centers[current_high_start] - bin_size / 2.0
                    end_time = bin_centers[i - 1] + bin_size / 2.0
                    high_intervals.append((start_time, end_time))
                current_high_start = None
            # Start or continue a low interval.
            if current_low_start is None:
                current_low_start = i
        else:
            # If we are in a low interval, close it.
            if current_low_start is not None:
                if (i - current_low_start) >= min_consecutive_bins:
                    start_time = bin_centers[current_low_start] - bin_size / 2.0
                    end_time = bin_centers[i - 1] + bin_size / 2.0
                    low_intervals.append((start_time, end_time))
                current_low_start = None
            # Start or continue a high interval.
            if current_high_start is None:
                current_high_start = i

    # Handle any ongoing interval at the end of the array.
    if current_low_start is not None and (len(ripple_rate) - current_low_start) >= min_consecutive_bins:
        start_time = bin_centers[current_low_start] - bin_size / 2.0
        end_time = bin_centers[-1] + bin_size / 2.0
        low_intervals.append((start_time, end_time))
        
    if current_high_start is not None and (len(ripple_rate) - current_high_start) >= min_consecutive_bins:
        start_time = bin_centers[current_high_start] - bin_size / 2.0
        end_time = bin_centers[-1] + bin_size / 2.0
        high_intervals.append((start_time, end_time))
        
    return low_intervals, high_intervals


In [ ]:
bin_size =1
min_consecutive_bins = 1# or adjust as needed
pre_low_intervals,pre_high_intervals = get_ripple_intervals(pre_timestamps, pre_ripple_rate, bin_size,
                                             threshold=0.1, min_consecutive_bins=min_consecutive_bins)
mid_low_intervals,mid_high_intervals  = get_ripple_intervals(mid_timestamps, mid_ripple_rate, bin_size,
                                             threshold=0.1, min_consecutive_bins=min_consecutive_bins)
post_low_intervals,post_high_intervals = get_ripple_intervals(post_timestamps, post_ripple_rate, bin_size,
                                              threshold=0.1, min_consecutive_bins=min_consecutive_bins)
all_low_intervals = pre_low_intervals + mid_low_intervals + post_low_intervals

In [ ]:

# Compute total durations
def total_duration(intervals):
    """Compute total duration from a list of [start, end] intervals."""
    return sum(end - start for start, end in intervals)

# Low ripple durations
pre_low_duration = total_duration(pre_low_intervals)
mid_low_duration = total_duration(mid_low_intervals)
post_low_duration = total_duration(post_low_intervals)
all_low_intervals = pre_low_intervals + mid_low_intervals + post_low_intervals
all_low_duration = total_duration(all_low_intervals)

# High ripple durations
pre_high_duration = total_duration(pre_high_intervals)
mid_high_duration = total_duration(mid_high_intervals)
post_high_duration = total_duration(post_high_intervals)
all_high_intervals = pre_high_intervals + mid_high_intervals + post_high_intervals
all_high_duration = total_duration(all_high_intervals)

# Print results

def match_interval_durations(low_intervals, high_intervals):
    """Trim the longer list of intervals so both have the same total duration."""
    def trim_intervals(intervals, target_duration):
        trimmed = []
        total = 0
        for start, end in intervals:
            duration = end - start
            if total + duration <= target_duration:
                trimmed.append((start, end))
                total += duration
            else:
                # Clip the final interval to fit exactly
                needed = target_duration - total
                if needed > 0:
                    trimmed.append((start, start + needed))
                    total += needed
                break
        return trimmed

    dur_low = total_duration(low_intervals)
    dur_high = total_duration(high_intervals)

    if dur_low > dur_high:
        low_intervals = trim_intervals(low_intervals, dur_high)
    elif dur_high > dur_low:
        high_intervals = trim_intervals(high_intervals, dur_low)

    return low_intervals, high_intervals
# For pre
pre_low_intervals, pre_high_intervals = match_interval_durations(pre_low_intervals, pre_high_intervals)

# For mid
mid_low_intervals, mid_high_intervals = match_interval_durations(mid_low_intervals, mid_high_intervals)

# For post
post_low_intervals, post_high_intervals = match_interval_durations(post_low_intervals, post_high_intervals)

# === Print durations ===

all_low_intervals  = pre_low_intervals + mid_low_intervals + post_low_intervals
all_high_intervals = pre_high_intervals + mid_high_intervals + post_high_intervals
import os
import pandas as pd
import numpy as np
pre_duration= total_duration(pre_low_intervals)
mid_duration= total_duration(mid_low_intervals)
post_duration= total_duration(post_low_intervals)

In [ ]:
import os
import pandas as pd
from criticality.av_analysis import (
        coarse_grain_avalanche_detection_all_units_intervals,
        plot_power_law_ccdf,
        plot_dcc,
        compute_scaling_with_avg
    )
def run_dcc_pipeline_forplot(
    st_time,
    bin_size,
    pre_low_intervals,
    pre_high_intervals,
    xmins_size,
    xmaxs_size,
    xmins_dur,
    xmaxs_dur,
    animal_name,
    day,
    sleep_idx,
    animal,
    brain_region,
    cell_type,
    base_dir,
    csv_path_dcc=None,
    csv_path_beta=None,
    plot_scaling=True
):
    from criticality.av_analysis import (
        coarse_grain_avalanche_detection_all_units_intervals,
        plot_power_law_ccdf,
        plot_dcc,
        compute_scaling_with_avg
    )

    # Avalanche detection
    # ---- data (../csv) ----
    _a = read_csv('avalanches_low_swr_rate.csv')
    aval_sizes_low, aval_durs_low = _a['aval_sizes_low'].to_numpy(), _a['aval_durs_low'].to_numpy()
    _a = read_csv('avalanches_high_swr_rate.csv')
    aval_sizes_high, aval_durs_high = _a['aval_sizes_high'].to_numpy(), _a['aval_durs_high'].to_numpy()
    # ---- the same from the recordings, with analysis_scripts/: ----
    # aval_sizes_low, aval_durs_low, *_ = coarse_grain_avalanche_detection_all_units_intervals(
    #     st_time, bin_size, pre_low_intervals)
    # aval_sizes_high, aval_durs_high, *_ = coarse_grain_avalanche_detection_all_units_intervals(
    #     st_time, bin_size, pre_high_intervals)
 
    # Prepare data dictionaries
    data_dict_sizes = {'low_ripple': aval_sizes_low, 'high_ripple': aval_sizes_high}
    data_dict_durations = {'low_ripple': aval_durs_low, 'high_ripple': aval_durs_high}
    data_dict = {'low_ripple': (aval_sizes_low, aval_durs_low), 'high_ripple': (aval_sizes_high, aval_durs_high)}

    # Plot power-law CCDF for size
    title_sizes = f"{animal_name}_{day}_{brain_region}_{cell_type}_{sleep_idx}_Avalanche Size CCDF"
    plot_file_name = f"figS2_E_av_size_{animal_name}_{day}_{brain_region}_{cell_type}_{sleep_idx}_lh_ripples.svg"
    save_path_size = os.path.join(base_dir, plot_file_name)
    alpha_size_dict = plot_power_law_ccdf(data_dict_sizes, xmins_size, xmaxs_size,
                                          "Avalanche Size", "CCDF", title_sizes,
                                          animal_name, day, animal, sleep_idx,
                                          save_path_svg=save_path_size, csv_path=None, brain_region=brain_region)

    # Plot power-law CCDF for durations
    title_durations = f"{animal_name}_{day}_{brain_region}_{cell_type}_{sleep_idx}_Avalanche Duration CCDF"
    plot_file_name = f"figS2_E_av_durations_{animal_name}_{day}_{brain_region}_{cell_type}_{sleep_idx}_lh_ripples.svg"
    save_path_dur = os.path.join(base_dir, plot_file_name)
    alpha_duration_dict = plot_power_law_ccdf(data_dict_durations, xmins_dur, xmaxs_dur,
                                              "Avalanche Duration", "CCDF", title_durations,
                                              animal_name, day, animal, sleep_idx,
                                              save_path_svg=save_path_dur, csv_path=None, brain_region=brain_region)

    # Compute scaling slope β from mean size vs duration
    title_scaling = "Avalanche Size-Duration Relationship"
    plot_file_name = f"figS2_E_Aav_scaling_law_{animal_name}_{day}_{brain_region}_{sleep_idx}_lh_ripples.svg"
    save_path_scaling = os.path.join(base_dir, plot_file_name) if plot_scaling else None
    obs_beta_dict = compute_scaling_with_avg(data_dict, "Duration", "Size", title_scaling,
                                             animal_name, day, sleep_idx, animal,
                                             save_path_svg=save_path_scaling,
                                             csv_path=csv_path_beta,
                                             brain_region=brain_region)

    # Compute DCC
    dcc_dict = {}
    for label in alpha_size_dict.keys():
        alpha_size = alpha_size_dict[label]
        alpha_dur = alpha_duration_dict[label]
        obs_beta = obs_beta_dict[label]
        beta_predicted = (alpha_dur - 1) / (alpha_size - 1)  # crackling noise scaling
        dcc = abs(obs_beta - beta_predicted)
        dcc_dict[label] = {
            "beta_predicted": beta_predicted,
            "obs_beta": obs_beta,
            "dcc": dcc
        }
    return dcc_dict

In [ ]:
run_dcc_pipeline_forplot(
    st_time=st.time,
    bin_size=0.05,
    pre_low_intervals=pre_low_intervals,
    pre_high_intervals=pre_high_intervals,
    xmins_size=[90,0],
    xmaxs_size=[0, 0],
    xmins_dur=[10, 0],
    xmaxs_dur=[0, 0],
    animal_name=animal_name,
    day=day,
    sleep_idx=0,
    animal=animal,
    brain_region=brain_region,
    cell_type=cell_type,
    base_dir=r"../figure_output"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
import os, re, glob, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
# ---------- Summary stats ----------
import os, glob, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# Global font: prefer Arial, with safe fallbacks
mpl.rcParams['svg.fonttype'] = 'none' 
mpl.rcParams['text.usetex'] = False
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [
    'Helvetica Neue', 'Helvetica', 'Arial', 'Liberation Sans', 'DejaVu Sans'
]

# Mathtext to match (works if those faces exist on your system)
mpl.rcParams['mathtext.fontset'] = 'custom'
mpl.rcParams['mathtext.rm'] = 'Helvetica Neue'   # or 'Helvetica'
mpl.rcParams['mathtext.it'] = 'Helvetica Neue:italic'
mpl.rcParams['mathtext.bf'] = 'Helvetica Neue:bold'

# (optional) sizes + minus sign
mpl.rcParams['font.size'] = 16
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams['axes.unicode_minus'] = False

# Global sizes (tweak to taste)
mpl.rcParams['font.size'] = 14
mpl.rcParams['axes.titlesize'] = 14
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 14
mpl.rcParams['ytick.labelsize'] = 14
mpl.rcParams['legend.fontsize'] = 14
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
mpl.rcParams["axes.linewidth"] = 1.5       # spine width
mpl.rcParams["xtick.major.width"] =1.5
mpl.rcParams["ytick.major.width"] = 1.5
mpl.rcParams["xtick.minor.width"] = 1.5
mpl.rcParams["ytick.minor.width"] = 1.5
# Load both CSV files for alpha
csv_path_alpha_size = r"D:\Documents\GitHub\Sleep_criticaltiy\data\l_h_ripples\log\av_size_all_sleep.csv"
csv_path_alpha_duration = r"D:\Documents\GitHub\Sleep_criticaltiy\data\l_h_ripples\log\av_duration_all_sleep.csv"

# ---- data (../csv) ----
df_size = read_csv('av_size_all_sleep.csv')
df_duration = read_csv('av_duration_all_sleep.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# df_size = pd.read_csv(csv_path_alpha_size)
# df_duration = pd.read_csv(csv_path_alpha_duration)

# ---------------------------
# 2. Filter for sleep_idx == 0 or 4.
# ---------------------------
# 3. Check that each dataset has exactly 2 labels.
# ---------------------------
labels_size = df_size['label'].unique()
labels_duration = df_duration['label'].unique()

assert len(labels_size) == 2, f"Expected exactly 2 labels for size, got {labels_size}"
assert len(labels_duration) == 2, f"Expected exactly 2 labels for duration, got {labels_duration}"

# ---------------------------
# 4. Function to pivot the DataFrame for a given measure ('alpha' or 'ks_stat').
# ---------------------------
def create_pivot(df, measure):
    # Pivot with index consisting of animal_name, day, and sleep_idx.
    pivot_df = df.pivot_table(
        index=['animal_name', 'day', 'sleep_idx'],
        columns='label',
        values=measure
    ).dropna()
    return pivot_df

# Create pivot tables for both measures and for both CSV files.
# For avalanche size.
pivot_size_alpha = create_pivot(df_size, 'alpha')
pivot_size_ks    = create_pivot(df_size, 'ks_stat')

# For avalanche duration.
pivot_duration_alpha = create_pivot(df_duration, 'alpha')
pivot_duration_ks    = create_pivot(df_duration, 'ks_stat')

# ---------------------------
# 5. Group the pivot data by (animal_name, day), averaging if needed.
# This provides one value per (animal, day) pair.
# ---------------------------
grouped_size_alpha = pivot_size_alpha.reset_index().groupby(['animal_name', 'day', 'sleep_idx']).mean()
grouped_duration_alpha = pivot_duration_alpha.reset_index().groupby(['animal_name', 'day', 'sleep_idx']).mean()
grouped_size_ks = pivot_size_ks.reset_index().groupby(['animal_name', 'day', 'sleep_idx']).mean()
grouped_duration_ks = pivot_duration_ks.reset_index().groupby(['animal_name', 'day', 'sleep_idx']).mean()

# ---------------------------
# 6. Perform paired t-tests on the grouped data.
# ---------------------------
# For alpha measure.
from scipy.stats import wilcoxon
import numpy as np
from scipy.stats import wilcoxon

def paired_wilcoxon_with_n(x, y):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    mask = np.isfinite(x) & np.isfinite(y)
    n_sessions = mask.sum()
    
    if n_sessions < 2:
        return np.nan, np.nan, n_sessions
    
    stat, p = wilcoxon(x[mask], y[mask])
    return stat, p, n_sessions

# --- Alpha (power-law exponent) ---
w_stat_size_alpha, p_val_size_alpha, n_size_alpha = paired_wilcoxon_with_n(
    grouped_size_alpha["low_ripple"],
    grouped_size_alpha["high_ripple"]
)

w_stat_duration_alpha, p_val_duration_alpha, n_duration_alpha = paired_wilcoxon_with_n(
    grouped_duration_alpha["low_ripple"],
    grouped_duration_alpha["high_ripple"]
)

# --- KS statistic ---
w_stat_size_ks, p_val_size_ks, n_size_ks = paired_wilcoxon_with_n(
    grouped_size_ks["low_ripple"],
    grouped_size_ks["high_ripple"]
)

w_stat_duration_ks, p_val_duration_ks, n_duration_ks = paired_wilcoxon_with_n(
    grouped_duration_ks["low_ripple"],
    grouped_duration_ks["high_ripple"]
)


print("Paired Wilcoxon signed-rank test results (grouping by animal_name and day):")
print(f"Avalanche Size - Alpha:     p = {p_val_size_alpha:.2e}")
print(f"Avalanche Duration - Alpha: p = {p_val_duration_alpha:.2e}")
# ---------------------------
# 7. Melt the grouped data for plotting.
# ---------------------------
def melt_grouped(grouped_df, measure):
    # grouped_df has index ['animal_name', 'day'] and columns (labels)
    return grouped_df.reset_index().melt(id_vars=['animal_name', 'day', 'sleep_idx'],
                                         value_vars=["low_ripple", "high_ripple"],
                                         var_name='Label',
                                         value_name=measure)

melted_size_alpha = melt_grouped(grouped_size_alpha, 'alpha')
melted_duration_alpha = melt_grouped(grouped_duration_alpha, 'alpha')
melted_size_ks = melt_grouped(grouped_size_ks, 'ks_stat')
melted_duration_ks = melt_grouped(grouped_duration_ks, 'ks_stat')

# ---------------------------
# 8. Plotting: 2 rows x 2 columns.
# Row 0: Avalanche Size and Duration for "alpha"
# Row 1: Avalanche Size and Duration for "ks_stat"
# The boxplot order is explicitly set to display "low_ripple" first.
# ---------------------------
import matplotlib.pyplot as plt
import seaborn as sns

## 1x2 grid (one row, two plots)
fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=False)

palette = {"low_ripple": "tab:blue", "high_ripple": "tab:red"}
order = ["low_ripple", "high_ripple"]

# Avalanche Size (left)
sns.boxplot(
    data=melted_size_alpha, x="Label", y="alpha",
    palette=palette, order=order,
    showfliers=False, width=0.6, linewidth=2,
    ax=axes[0]
)
axes[0].set_title("Avalanche Size (S)")
axes[0].set_ylabel("Power-law exponent (τ)")
axes[0].grid(False)
axes[0].spines['top'].set_visible(False)   # Remove top spine
axes[0].spines['right'].set_visible(False) # Remove right spine
# Avalanche Duration (right)
sns.boxplot(
    data=melted_duration_alpha, x="Label", y="alpha",
    palette=palette, order=order,
    showfliers=False, width=0.6, linewidth=2,
    ax=axes[1]
)
axes[1].set_title("Avalanche Duration (T)")
axes[1].set_ylabel("Power-law exponent (α)")
axes[1].grid(False)
axes[1].spines['top'].set_visible(False)   # Remove top spine
axes[1].spines['right'].set_visible(False) # Remove right spine
plt.tight_layout()

plt.savefig(r"../figure_output\S2_E_low_high_ripple_avalanche_fit_bar_equalsample.svg")  # SVG is vector-based
plt.show()
# Compute and print median values for each condition
for title, (df_name, col_name) in {
    "Avalanche Size (alpha)": ("melted_size_alpha", "alpha"),
    "Avalanche Duration (alpha)": ("melted_duration_alpha", "alpha")}.items():
    df = globals()[df_name]  # Get DataFrame from variable name
    medians = df.groupby("Label")[col_name].median()
    for label, median in medians.items():
        pass
# Pick any one of them (they all have the same session structure)
g = grouped_size_alpha.reset_index()

# ---- number of total sessions ----
n_sessions = g[["animal_name", "day", "sleep_idx"]].drop_duplicates().shape[0]

# ---- number of unique animals ----
n_animals = g["animal_name"].nunique()

print(f"Total sessions = {n_sessions}")
print(f"Total animals  = {n_animals}")
# Load the CSV
csv_path = r"D:\Documents\GitHub\R_Project\PPP\codebook\Criticality_analysis\avalanche\dcc_all_all_sleep.csv"
# ---- data (../csv) ----
df = read_csv('dcc_all_all_sleep.csv')
# ---- the same from the recordings, with analysis_scripts/: ----
# df = pd.read_csv(csv_path)


# Check the labels
labels = df['label'].unique()
assert len(labels) == 2, f"Expected exactly 2 labels, got {labels}"

# Reshape the data into wide format: one row per animal-day, two columns for each label
pivot_df = df.pivot_table(
    index=['animal_name', 'day', 'sleep_idx'],
    columns='label',
    values='dcc'
).dropna()  # Drop any rows that don't have both labels

# Perform paired t-test
from scipy.stats import wilcoxon
import numpy as np

# Extract paired values and remove NaNs safely
x = pivot_df[labels[0]].to_numpy(float)
y = pivot_df[labels[1]].to_numpy(float)

mask = np.isfinite(x) & np.isfinite(y)
x = x[mask]
y = y[mask]

w_stat, p_val = wilcoxon(x, y)

print(f"\nWilcoxon signed-rank test between {labels[0]} and {labels[1]}:")
print(f"p = {p_val:.4e}, n = {len(x)}")

# Melt for boxplot
melted_df = pivot_df.reset_index().melt(id_vars=['animal_name', 'day'], 
                                        value_vars=labels,
                                        var_name='Label',
                                        value_name='DCC')

# Plot boxplot
plt.figure(figsize=(4, 5))
palette = {"low_ripple": sns.color_palette("Set1")[1], "high_ripple": sns.color_palette("Set1")[0]}
ax = sns.boxplot(data=melted_df, x='Label', y='DCC', palette=palette, showfliers=False, width=0.6,linewidth=2)

# Customize appearance
ax.grid(False)  # Remove grid
ax.spines['top'].set_visible(False)   # Remove top spine
ax.spines['right'].set_visible(False) # Remove right spine
ax.set_xlabel("") 
plt.tight_layout()
plt.savefig(r"../figure_output\S2_F_low_high_ripple_dcc.svg")  # SVG is vector-based
plt.show()